# DS4DS Exercise Sheet 05

**General Instructions:**

- Review the weekly course material (lectures, readings, slides, etc.) before starting the exercises.
- Complete all assigned exercises independently before the Q&A session.
- Please use Julia Version 1.12.x to ensure compatibility.
- Please only write between the `#--- YOUR CODE STARTS HERE ---#` and `#--- YOUR CODE ENDS HERE ---#` comments.

### Task 1: Regularization 

In the first task, the same data as already used in Ex04 task3 will be utilized.
The given data was produced using a polynomial of the form


$$
\begin{align}
    y = w_0 + x \cdot w_1 + \dots + x^p \cdot w_p
\end{align}
$$

with an unknown degree $p$.

Below you are given data for the inputs $x$ and the outputs $y$.  All of the data for the measurements $y$ is affected by bias-free, additive, Gaussian noise, i.e., the values you are given below do not fully follow the formula given above, but instead :

$$
\begin{align}
    y = w_0 + x \cdot w_1 + \dots + x^p \cdot w_p + \nu \quad \mathrm{with} \, \nu \sim \mathcal{N}(0, \sigma_n^2).
\end{align}
$$

The variance of the noise process is unknown, but it is also not necessary to estimate it for completing this task.


The task in the following is to identify the coefficents using the training data and validate the result using the validation data, like already done in Ex04. 
But in this task we will assume a fixed degree of the polynom $p = 6$ for our model. 

**Model:** We will make use of:

$$
\begin{align}
    y = w_0 + x \cdot w_1 + x^2 \cdot w_2 + x^3 \cdot w_3 + x^4 \cdot w_4 + x^5 \cdot w_5 + x^6 \cdot w_6 .
\end{align}
$$


In [36]:
import numpy as np
import scipy.io

In [37]:
data_1 = scipy.io.loadmat("data_task_3_ex04.mat");

# training data
x_training = data_1["x_training"].flatten();
y_training = data_1["y_training"].flatten();

# validation data
x_validation = data_1["x_validation"].flatten();
y_validation = data_1["y_validation"].flatten();


**a)** Based on the implementation of EX04, calculate the estimation error of an OLS estimator, which uses the model defined above.

**Hint**: The functions from Ex04 can be used here. 

In [38]:
def regressor_matrix(x, p):
	"""
	Args:
		x: Input data
		p: Degree of the polynomial

	Returns:
		Regressor matrix Z with shape (length(x), p+1)
	"""
	assert x.ndim == 1
	### BEGIN SOLUTION
	Z = np.zeros((len(x), p + 1))

	for i in range(p + 1):
		Z[:, i] = x ** i

	### END SOLUTION

	return Z


In [39]:
def compute_parameter_estimation(x, y, p):
	"""
	Args:
		x: Input data
		y: Output data
		p: Degree of the polynomial

	Returns:
		parameter vector w with shape (p+1)
	"""
	assert x.ndim == y.ndim == 1
	### BEGIN SOLUTION
	Z = regressor_matrix(x, p)
	w = np.linalg.inv(Z.T @ Z) @ Z.T @ y
	### END SOLUTION

	return w


In [40]:
def error_function(a, b):
	"""Computes the error between the two input vectors."""

	assert len(a) == len(b)
	assert a.ndim == b.ndim == 1

	N = len(a)
	return np.sqrt(1 / N * np.sum((a - b) ** 2))


To reduce the error, the ridge regression should be applied in the following subtasks. 

**b)** For a proper scaling, the data should be normalized in the beginning. Therefore, write a function which normalizes the given data vector $d$ based on the following equation:

$$
\begin{align}
    d_{norm} = \frac{d - \mu(d)}{\sigma(d)}
\end{align}
$$

where $\mu(d)$ defines the mean of the data $d$ and $\sigma(d)$ defines the standard deviation, accordingly.

There are two common mistakes made when data is normalized:

1. $\mu$ and $\sigma$ are calculated on the union of training and test (or val) data. 
*Problem:* When you normalize the training data, information from the test (or val) data is introduced to the training data by the usage of $\mu$ and $\sigma$. This introduces bias to your tests (or vals) later on such that the results are overestimating the general performance. Your model will perform artificially well on the test set because it already indirectly saw the test data's distribution.

2. Training and test (or val) data are normalized separately.
*Explanation:* Assume you have normalized the training data x_training, y_training with $\mu_{train}$ and $\sigma_{train}$ and trained an algorithm A on it. Now you want to know, how A would be doing if it is applied on the actual problem the data comes from. This is done by emulating the actual problem. In a real application of the trained A, you get some $\hat{x}$, normalize it with $\mu_{train}$ and $\sigma_{train}$, and apply A on it to get a prediction $\hat{y}_{pred}$ (within normalization space). This is mimiced in testing (or validation), i.e. you use $\mu_{train}$ and $\sigma_{train}$ to normalize your test (or val) data. Many people do a mistake here and calculate a separate $\mu_{test}$ and $\sigma_{test}$, which does not represent the real application any more and introduces bias. A sufficient way of thinking about normalization is, that it is part of your actual algorithm A with parameters $\mu_{train}$ and $\sigma_{train}$. If a specific voltage reading means "danger" in the training set, standardizing the test set with a different mean might map that exact same physical voltage to a completely different normalized number, breaking your algorithm's logic.

*Coming back to the assignment:* A common name for a normalizer is *StandardScaler*. There you store your $\mu_{train}$ and $\sigma_{train}$. The StandardScaler is used for two function. You see them below with further explanations. You are supposed to fill in the missing parts.

In [41]:
# Nothing to do in this cell!

class StandardScaler:
    """
    Standardizes the input data by removing the mean and scaling to unit variance.
    """
    def __init__(self):
        self.mean_ = None
        self.scale_ = None

    def fit(self, x):
        """
        Calculates and stores the mean and standard deviation of the given data vector X.
        """
        # Convert to numpy array to ensure math operations work smoothly
        X = np.array(x, dtype=float)
        self.mean_ = np.mean(X)
        self.scale_ = np.std(X, ddof=1) # Use sample standard deviation (ddof=1) for an unbiased estimator - divides by N-1

    def transform(self, x):
        """
        Transforms the data using the fitted mean and std.
        """
        # Check if the scaler has been fitted yet
        if self.mean_ is None or self.scale_ is None:
            raise ValueError("The scaler has not been fitted. Please call fit() first.")
        
        X = np.array(x, dtype=float)

        return (X - self.mean_) / self.scale_

In [42]:
# test = convert(Vector{Float64}, collect(1:5))
test_data = [1.0, 2.0, 3.0, 4.0, 5.0]

# SS = StandardScaler()
SS = StandardScaler()

# fit!(SS, test)
SS.fit(test_data)

# test_norm = transform(SS, test)
test_norm = SS.transform(test_data)

truth = [-1.2649110640673518, -0.6324555320336759, 0.0, 0.6324555320336759, 1.2649110640673518]

# Assertions to verify the output matches the expected truth
assert len(test_norm) == len(test_data), "Size mismatch"
assert np.allclose(test_norm, truth), "Transformed values do not match the expected truth"

print("All tests passed! Transformed data:")
print(test_norm)

All tests passed! Transformed data:
[-1.26491106 -0.63245553  0.          0.63245553  1.26491106]


**c)** Implement the function below to calculate the parameters $\mathbf{w_{ridge}}$ using the ridge regression approach from the lecture.
The function should be able to generate a regressor matrix $\mathbf{Z}$ from the input vector $\mathbf{x}$ using the `regressor_matrix(x, p)` function implemented in subtask a) based on the model of a polynomial of degree $p$.
This regressor matrix $Z$ is then to be used together with the measurement vector $\mathbf{y}$ and the penatly term lambda to estimate the parameters $\mathbf{w_{ridge}}$ based on the ridge regression approach from the lecture.

$$\mathbf{w}_{ridge} = (\mathbf{Z}^T \mathbf{Z} + \lambda \mathbf{I})^{-1} \mathbf{Z}^T \mathbf{y}$$

In [43]:
def compute_parameter_estimation_ridge(x, y, p, lambda_val):
	"""
	Args:
		x: Input data
		y: Output data
		p: Degree of the polynomial
		lambda_val: scalar weighting factor

	Returns:
		parameter vector w_ridge with shape (p+1)
	"""
	assert x.ndim == y.ndim == 1
	assert np.isscalar(p) and np.isscalar(lambda_val), "p and lambda must be scalars"
	### BEGIN SOLUTION
	Z = regressor_matrix(x, p)
	I = np.eye(p + 1)
	w_ridge = np.linalg.inv(Z.T @ Z + lambda_val * I) @ Z.T @ y
	### END SOLUTION

	return w_ridge


**d)** Use the StandardScaler from b) to normalize x_training and y_training, afterwards use the implemented function in c) to evaluate the estimation error for different penalty factors lambda applying the ridge regression.

*Notes:*
- Consider only the given lambda values.
- Calculate the error with our error_function from above per lambda and store the minimum error among all identified models in the variable err_min.
- A polynomial of of degree $p = 6$ is to be used as the model.
- One StandardScaler for x_training and one for y_training.
- The error should be calculated in normalization space (i.e. compare y_pred to the normalized y_validation).


In [44]:
p = 6 # degree of polynomial
lambda_vec = [0, 1/16, 1/8, 1/4, 1/2, 1, 2, 4, 8, 16] # lambdas to be evaluated
best_lambda = None  # initialize best lambda

err_min = np.nan  # initialize minimal error
SS_x = StandardScaler()  # StandardScaler for x-data.
SS_y = StandardScaler()  # StandardScaler for y-data.



### BEGIN SOLUTION
SS_x.fit(x_training)
X_norm = SS_x.transform(x_training)

SS_y.fit(y_training)
Y_norm = SS_y.transform(y_training)

for i in range(len(lambda_vec)):
    lambda_val = lambda_vec[i]
    w_ridge = compute_parameter_estimation_ridge(X_norm, Y_norm, p, lambda_val)
    Z_validation = regressor_matrix(SS_x.transform(x_validation), p)
    y_pred = Z_validation @ w_ridge
    y_pred_original_scale = SS_y.mean_ + y_pred * SS_y.scale_
    err = error_function(y_pred_original_scale, y_validation)
    if np.isnan(err_min) or err < err_min:
        best_lambda = lambda_val
        err_min = err

### END SOLUTION

print("#######################")
print("Minimal error: ", err_min)
print("Best lambda: ", best_lambda)


#######################
Minimal error:  21.083913107836818
Best lambda:  2


### Task 2: Weighted least squares 
#### Estimation of physical parameters of the barometric altitude formula:

The weighted least squares (WLS) approach is to be used to estimate the physical parameters $p_0$ and $\alpha$ of the barometric altitude formula

$$
\begin{align}
    p(h) = p_0 e^{\alpha \cdot h}.
\end{align}
$$

The parameter $p_0$ describes the air pressure at sea level and the parameter $\alpha$ is a constant that describes the exponential decrease in air pressure. 
Using the barometric altitude formula, the air pressure $p$ is defined as a function of the height $h$ above sea level.
To determine the parameters $p_0$ and $\alpha$, pressure measurements $p$ are available as a function of the height $h$. 
These were determined using 10 weather balloons with a pressure sensor and a line on which the balloon's ascent height can be varied.
The transmission of the pressure data at an altitude $h$ was transmitted to the ground station using an analog radio signal.
This radio signal is superimposed with a noise, whereby the variance of the noise increases with increasing altitude $h$ due to the greater
distance to the ground station.

<img src="weather-balloon.png" alt="measurements" width="400"/>

[(Picture source)](https://www.farmersalmanac.com/weather-balloons-19447)

The measured pressure values were recorded at altitudes of 150 m, 200 m, 250 m, ... , 2000 m (38 equidistant heights). 
The pressure measurements were realized in the unit $\text{Pa} = \frac{\text{N}}{\text{m}^2}$.

The results of the above-mentioned experiment are stored in the data set `data_task_2.mat`.
It contains a measurement matrix $p$ of the pressure and $h$ of the height. 
These measurement matrices have the dimension 38 × 10, whereby the rows each contain measured values of a constant height.



In [45]:
data_2 = scipy.io.loadmat("data_task_2.mat");

# 2. Extract the variables 'p' and 'h'
# scipy.io.loadmat returns a dictionary, so we access them by their string keys
p = data_2["p"]
h = data_2["h"]

# 3. Convert them into flat 1D vectors
# In Julia, `reduce(vcat, x)` is often used to squash multidimensional arrays into a single vector.
# In Python's NumPy, the .flatten() method does exactly this.
p_vec = p.flatten()
h_vec = h.flatten()

**a)** The parameters occur non-linearly in the barometric altitude formula. Through which mathematical transformation can this non-linear estimation problem be transformed into a linear one such that linear LS (Least-Squares) can be applied? Write a function which implements this transformation to prepare the data for the use of linear LS methods.

The output of the function should be the regressor vector Z and the measurement vector Y, which can be used in linear LS methods 

The barometric altitude formula is non-linear:
$$p_{meas} = p_0 \exp(-\alpha h)$$
To apply Ordinary Least Squares (OLS), we must transform it into a linear equation of the form $y = \mathbf{z}^T \mathbf{w}$. We do this by taking the natural logarithm of both sides:
$$\ln(p_{meas}) = \ln(p_0) - \alpha h$$


**Hint**: Use the provided, already concatenated vectors in this task.

In [46]:
import numpy as np

def prepare_data(p_meas, h_meas):
    """
    Args:
        p_meas: Measured pressure data (1D array)
        h_meas: Measured height data (1D array)
    Returns:
        Y: Measurement vector for linear LS
        Z: Regressor matrix for linear LS
    """
    ### BEGIN SOLUTION
    
    # Transform the non-linear pressure measurements using natural log
    Y = np.log(p_meas)
    
    # Build the regressor matrix with a column of 1s (for ln(p_0)) and a column of h (for -alpha)
    Z = np.column_stack((np.ones(len(h_meas)), h_meas))
    
    ### END SOLUTION
    
    return Y, Z

**b)** Write a function to calculate the parameter vector $w$ that corresponds to $p_0$ and $\alpha$ within transformed space of the barometric height equation for the given data set. To do this, you can use the OLS-function from Ex04.

In [47]:
def parameter_calculation_OLS(Z, Y):
	"""
	Args:
		Z: regressor matrix with shape (n_measurements, n_regressors)
		Y: measurement_vector with shape (n_measurements)

	Returns:
		The parameter vector w with shape (n_parameters)
	"""

	### BEGIN SOLUTION
	w = np.linalg.inv(Z.T @ Z) @ Z.T @ Y
	### END SOLUTION
	return w


Since the measurement's variance is increasing with the height, in the following the WLS from the lecture should be applied to reduce the estimation error even further.

**c)** 
Write a function which returns the weighting matrix W and covariance matrix R for WLS. 

*Hint 1:* Do not forget the actual number of data points we have.


In [53]:
def calculate_W(p_matrix):
    """
    Args:
        p_matrix: 2D array of measurements (rows = heights, columns = readings)
    Returns:
        W: Weighting matrix 
        R: Covariance matrix 
    """
    ### BEGIN SOLUTION
    
    # 1. Calculate the variance for each height (along the rows / axis=1).
    # We use ddof=1 to get the unbiased sample variance.
    variances = np.var(p_matrix, axis=1, ddof=1)

    # 2. Find out how many sensor readings were taken at each height
    n_readings = p_matrix.shape[1] 
    
    # 3. CRITICAL FIX: Repeat each variance to match the flattened data!
    # This turns our 38 variances into 380 variances, perfectly aligning 
    # with how p_vec and h_vec were flattened earlier.
    expanded_variances = np.repeat(variances, n_readings)
    
    # 4. R is a diagonal matrix containing these variances on its main diagonal.
    # (Assuming the noise between different heights is independent)
    R = np.diag(expanded_variances)
    
    # 5. W is the inverse of the covariance matrix.
    # For a diagonal matrix, this is exactly the same as 1 / variance for each element.
    W = np.linalg.inv(R)
    
    ### END SOLUTION
    
    return W, R

**d)** Use the calculated weigthing matrix to apply the WLS, like presented in the lecture, to estimate the weight vector $w$ that corresponds to the parameters for $p_0$ and $\alpha$ within transformed space. 
Therefore, write a function (below) which takes the regressor matrix $Z$ and  measurement vector $y$ provided by the prepare_data-function from task a), as well as the weighting matrix $W$ calculated with the function implemented in task c)

$$\mathbf{w}_{wls} = (\mathbf{Z}^T \mathbf{W} \mathbf{Z})^{-1} \mathbf{Z}^T \mathbf{W} \mathbf{y}$$

In [51]:
def parameter_calculation_WLS(Z, Y, W):
    """
    Args:
        Z: regressor matrix
        Y: measurement vector
        W: weighting matrix
    Returns:
        w_wls: The WLS parameter vector
    """
    ### BEGIN SOLUTION
    
    Z_transpose = Z.T
    
    # (Z^T * W * Z)^-1
    inverse_term = np.linalg.inv(Z_transpose @ W @ Z)
    
    # Z^T * W * Y
    product_term = Z_transpose @ W @ Y
    
    # Combine to solve for w
    w_wls = inverse_term @ product_term
    
    ### END SOLUTION
    
    return w_wls

In [54]:

### BEGIN SOLUTION

# 1. Prepare the data (linearize the equation and build Z and Y)
# Assuming p_vec (1D pressure) and h_vec (1D height) were loaded earlier
Y, Z = prepare_data(p_vec, h_vec)

# 2. Calculate the Weighting Matrix W
# Assuming p_matrix (2D pressure matrix) was loaded earlier
W, R = calculate_W(p)

# 3. Estimate the parameter vector using WLS
w_wls = parameter_calculation_WLS(Z, Y, W)

# 4. Extract the physical parameters from the transformed vector w
# Recall our transformation: w_0 = ln(p_0)  and  w_1 = -alpha

w_0 = w_wls[0]
w_1 = w_wls[1]

# Reverse the natural log using the exponential function (e^x)
p_0_estimated = np.exp(w_0)

# Reverse the negative sign
alpha_estimated = -w_1

### END SOLUTION

print("--- WLS Parameter Estimation Results ---")
print(f"w_0 (ln(p_0)): {w_0:.5f}")
print(f"w_1 (-alpha) : {w_1:.6f}")
print("-" * 38)
print(f"Estimated p_0   : {p_0_estimated:.2f} hPa")
print(f"Estimated alpha : {alpha_estimated:.6f} 1/m")

--- WLS Parameter Estimation Results ---
w_0 (ln(p_0)): 11.52673
w_1 (-alpha) : -0.000117
--------------------------------------
Estimated p_0   : 101390.22 hPa
Estimated alpha : 0.000117 1/m


### Task 3: Recursive least squares 
The [Worldwide Harmonised Light Vehicles Test Procedure](https://en.wikipedia.org/wiki/Worldwide_Harmonised_Light_Vehicles_Test_Procedure) (WLTP) is used to determine, e.g., fuel consumption, CO2 emission, ... of traditional and hybrid cars.
The profile is shown in the figure below.

<img src="WLTP.png" alt="measurements" width="400"/>

[source: Skript Antriebe für umweltfreundliche Fahrzeuge, UPB](https://ei.uni-paderborn.de/lea/lehre/veranstaltungen/lehrangebote/antriebe-fuer-umweltfreundliche-fahrzeuge)

The shown velocity profile is used in an experiment on the street, while the power needed to drive a car fullfilling the WLTP profile is measured.

The task in the following is to identify uncertain and varying model parameters based on measurement data.
We will use the same model already intoduced in the lecture, to calculate the power required for a certain velocity:

$$
\begin{align}
    P[k] = m \cdot g \cdot C_\text{r} \cdot v[k] + \nu \cdot v[k]^2 + \frac{ρ A C_\text{d}}{2} \cdot v[k]^3 .
\end{align}
$$

The measured power based on the equation above can be seen in the following figure.

<img src="WLTP_power.png" alt="measurements" width="400"/>

The data "WLTP_rain.mat", which is provided, was obtained during a rainy day. As the tire-street contact is highly depending on the street's wetness level, the rolling resistance coefficient $C_\text{r}$ changed during the test drive due to varying rain conditions.
To track how that parameter changed during the measurement, the recursive least squares (RLS) approach should be applied in this task.
The other parameters can be assumed to be not affected by the rain and, therefore, stay constant.

The Math: How RLS Works

Unlike Ordinary Least Squares (which calculates the parameters once using all available data in a giant batch), Recursive Least Squares (RLS) updates the parameter estimates step-by-step as new data streams in.

To do this, it calculates four things at every time step $k$:

1. The Error ($e_k$): How wrong was our previous model at predicting the new measurement?
$$e_k = y_k - \mathbf{z}_k \mathbf{w}_{k-1}$$

2. The Gain ($\mathbf{K}_k$): How much should we trust this new error? If our previous estimates were highly uncertain (large $\mathbf{P}$), the gain is high. The $\lambda$ is the "forgetting factor"—a value less than 1 tells the algorithm to slowly forget older data, allowing it to track parameters that change over time (like your tire resistance in the rain!).

$$\mathbf{K}_k = \frac{\mathbf{P}_{k-1} \mathbf{z}_k^T}{\lambda + \mathbf{z}_k \mathbf{P}_{k-1} \mathbf{z}_k^T}$$

3. The Parameter Update ($\mathbf{w}_k$): Adjust the parameters based on the error and the gain.
$$\mathbf{w}_k = \mathbf{w}_{k-1} + \mathbf{K}_k e_k$$

4. The Covariance Update ($\mathbf{P}_k$): Update our uncertainty matrix for the next step.
$$\mathbf{P}_k = \frac{1}{\lambda} \left( \mathbf{P}_{k-1} - \mathbf{K}_k \mathbf{z}_k \mathbf{P}_{k-1} \right)$$

In [55]:
data_WLTP_rain = scipy.io.loadmat("WLTP_rain.mat");
v_WLTP = data_WLTP_rain["v"].flatten();
P_WLTP = data_WLTP_rain["P"].flatten();

m = 1500; # mass of car in kg
g = 9.81; # gravity in m/s^2
density = 1.29; # density of air in kg/m^3
A = 2.0; # frontal area of car in m^2
Cd = 0.35; # drag coefficient
Cr_dry_street = 0.015; # rolling resistance coefficient
η = 4.5; # Bearing friction coefficient in kg/s

a, b, c = m * g * Cr_dry_street, η, (density * A * Cd) / 2 # condense parameters


**a)** Write a general function to calculate the parameters of a linear model using one step of the RLS shown in lecture.
The function should take the measurement and regressor value(s) and the covariance matrix and the parameter(s) of the last step k-1 as inputs and return the estimated parameter(s) and the new covariance matrix for the current stime step k.

In [56]:
import numpy as np

def parameter_calculation_RLS(Z, Y, lam, W_RLS_k_1, P_k_1):
    """
    Args:
        Z: regressor row vector of size (1, n)
        Y: measurement value (scalar)
        lam: forgetting factor (scalar)
        W_RLS_k_1: last parameter(s) of timestep k-1 of size (n, 1)
        P_k_1: last covariance matrix of size (n, n)
        
    Returns:
        W_RLS: New parameter vector of size (n, 1)
        P_k: New covariance matrix of size (n, n)
    """
    
    # Ensure matrices are explicitly 2D for NumPy dot products (@)
    # This prevents dimension crashing if 1D arrays are passed in.
    Z = np.atleast_2d(Z)
    W_RLS_k_1 = np.atleast_2d(W_RLS_k_1).reshape(-1, 1)
    P_k_1 = np.atleast_2d(P_k_1)
    
    ### BEGIN SOLUTION
    
    # 1. Compute the prediction error
    # Z @ W_RLS_k_1 results in a 1x1 matrix, so we use [0, 0] to extract the scalar
    error = Y - (Z @ W_RLS_k_1)[0, 0]
    
    # 2. Compute the Kalman Gain (K)
    Z_transpose = Z.T
    
    # The denominator is a scalar: lambda + (Z * P * Z^T)
    denominator = lam + (Z @ P_k_1 @ Z_transpose)[0, 0]
    
    # The gain K is a column vector (n, 1)
    K = (P_k_1 @ Z_transpose) / denominator
    
    # 3. Update the parameter vector
    W_RLS = W_RLS_k_1 + (K * error)
    
    # 4. Update the covariance matrix
    P_k = (1 / lam) * (P_k_1 - K @ Z @ P_k_1)
    
    ### END SOLUTION
    
    return W_RLS, P_k

In [57]:
# --- Code to check the implementation ---
num_steps = 11

# Z_test is 11x2, Y_test is 1 to 11
Z_test = np.ones((num_steps, 2))
Y_test = np.arange(1, num_steps + 1)

lambda_test = 0.5

# Initialize storage arrays
w_rls_test = np.ones((2, num_steps))
w_rls_test[:, 0] = [0.1, 0.2]  # Initial parameters at k=0 (index 0 in Python)

P_test = 10 * np.eye(2)        # Initial covariance matrix

# Loop through steps (indices 1 to 10 in Python)
for k in range(1, num_steps):
    # Z_test[k:k+1, :] slices a 1x2 row matrix safely
    Z_row = Z_test[k:k+1, :]
    Y_val = Y_test[k]
    W_prev = w_rls_test[:, k-1]
    
    W_new, P_test = parameter_calculation_RLS(Z_row, Y_val, lambda_test, W_prev, P_test)
    
    # Store the result
    w_rls_test[:, k] = W_new.flatten()

# Print final columns to verify against the truth arrays in your assignment
print("Final Parameters (Parameter A):")
print(w_rls_test[0, :])

print("\nFinal Parameters (Parameter B):")
print(w_rls_test[1, :])

Final Parameters (Parameter A):
[0.1        0.92926829 1.27355372 1.65871886 2.08003328 2.52868654
 2.99646965 3.47689431 3.96530732 4.45859302 4.95476895]

Final Parameters (Parameter B):
[0.2        1.02926829 1.37355372 1.75871886 2.18003328 2.62868654
 3.09646965 3.57689431 4.06530732 4.55859302 5.05476895]


**b)**  Use the implemented RLS function to estimate the  parameter $C_\text{r}$ and store the results in the given vector Cr_RLS.


**Hint1**: According to car model stated above, the RLS output data has to be further processed to provide an estimate of the parameter $C_\text{r}$ and NOT the parameter $a =  m \cdot g \cdot C_\text{r}$.

**Hint2**: During the first half of the measurement profile, the street was rather wet (rather high $C_\text{r}$) and during the second half comparatively dry (rather low $C_\text{r}$). You can use this observation to check your estimated $C_\text{r}$ on a qualitative basis.

$$P[k] = \underbrace{a \cdot v[k]}_{\text{Rolling}} + \underbrace{b \cdot v[k]^2}_{\text{Bearing}} + \underbrace{c \cdot v[k]^3}_{\text{Aero}}$$

Because we assume the bearing friction ($b$) and air resistance ($c$) stay constant despite the rain, we only want the RLS to estimate $a$ (which contains our rolling resistance $C_r$).

To do a 1-parameter RLS estimation, we have to subtract the known forces from our total measured power to isolate the unknown part.

. Effective Measurement ($Y$): $P[k] - b \cdot v[k]^2 - c \cdot v[k]^3$ <br>
. Regressor ($Z$): $v[k]$

In [58]:
# Assuming m, g, b, and c were defined in the previous cell
# m = 1500; g = 9.81
# b = eta; c = (rho * A * Cd) / 2

w_0 = 210  # Initial parameter of a = m * g * Cr
lam = 0.99 # Forgetting factor

num_steps = len(v_WLTP)

# Initialize storage arrays
Cr_RLS = np.zeros(num_steps)
W_RLS = np.zeros(num_steps)

# Set initial states at k=0
P_k = 10.0  
W_RLS[0] = w_0
Cr_RLS[0] = w_0 / (m * g)

### BEGIN SOLUTION
for k in range(1, num_steps):
    
    # 1. Extract the current velocity and measured power
    v_k = v_WLTP[k]
    P_meas_k = P_WLTP[k]
    
    # 2. Calculate the "Effective Measurement" (Y)
    # We subtract the known aerodynamic and bearing friction powers from the total power
    Y_k = P_meas_k - (b * v_k**2 + c * v_k**3)
    
    # 3. Define the Regressor (Z)
    Z_k = v_k
    
    # 4. Fetch the previous parameter and covariance
    W_prev = W_RLS[k-1]
    
    # 5. Run one step of Recursive Least Squares
    # (Your previous RLS function will handle the 1x1 matrix math)
    W_new, P_k = parameter_calculation_RLS(Z_k, Y_k, lam, W_prev, P_k)
    
    # 6. Extract the updated scalar parameter 'a'
    # Depending on how your RLS function returns 1x1 matrices, we extract the flat value
    a_estimated = np.array(W_new).flatten()[0]
    W_RLS[k] = a_estimated
    
    # 7. Convert 'a' into the actual Rolling Resistance Coefficient (Cr)
    # Since a = m * g * Cr, we simply divide by (m * g)
    Cr_RLS[k] = a_estimated / (m * g)
### END SOLUTION
Cr_RLS

array([0.01427115, 0.01427115, 0.01427115, ..., 0.01223952, 0.01223952,
       0.01223952], shape=(1801,))